# Hands-On Machine Learning with PyTorch (Aurelion Geron)

## 3) Deep Computer Vision using CNNs

In [ ]:
import numpy as np
import torch 
from sklearn.datasets import load_sample_images 

import torchvision.transforms.v2 as T

sample_images = np.stack(load_sample_images()["images"])
sample_images = torch.tensor(sample_images, dtype=torch.float32)/255

print(f"Shape of sample_images is {sample_images.shape}.")  # torch.Size([2, 427, 640, 3]

# Since PyTorch expects the channel dimension to preceed spatial dimensions, we need to permute 
sample_images_permuted = sample_images.permute(0, 3, 1, 2)
print(f"samples permuted have total shape {sample_images_permuted.shape}")  # torch.Size([2, 3, 427, 640])

cropped_images = T.CenterCrop((70, 120))(sample_images_permuted)
print(f"The cropped images have shape {cropped_images.shape}.") # torch.Size([2, 3, 70, 120])

Shape of sample_images is torch.Size([2, 427, 640, 3]).
samples permuted have total shape torch.Size([2, 3, 427, 640])
The cropped images have shape torch.Size([2, 3, 70, 120]).


### 1. Creating a 2D Convolutional Layer

In [8]:
import torch.nn as nn

torch.manual_seed(42)

in_channels = cropped_images.shape[1]
out_channels = 32 
kernel_size = 7

conv_layer = nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=kernel_size, padding="same") # stride, ...

fmaps = conv_layer(cropped_images)
print(f"fmaps, the result of convolutional layer, has shape {fmaps.shape}.") # torch.Size([2, 32, 70, 120])
print(f"The layer's weights have shape {conv_layer.weight.shape}")  # torch.Size([32, 3, 7, 7])
print(f"The layer's bias have shape {conv_layer.bias.shape}")   # torch.Size([32])

# Create max pooling layer 
max_pool = nn.MaxPool2d(kernel_size=2)  # stride defaults to kernel_size (here s=2)
print(f"Max pooling gives shape {max_pool(cropped_images).shape}")

# Create average pooling layer
avg_pool = nn.AvgPool2d(kernel_size=2)
print(f"Average pooling gives shape {avg_pool(cropped_images).shape}")

# Perform global average pooling
global_avg_pool = nn.AdaptiveAvgPool2d(output_size=1)
output = global_avg_pool(cropped_images)
print(f"Global average pooling gives {output}.")

fmaps, the result of convolutional layer, has shape torch.Size([2, 32, 70, 120]).
The layer's weights have shape torch.Size([32, 3, 7, 7])
The layer's bias have shape torch.Size([32])
Max pooling gives shape torch.Size([2, 3, 35, 60])
Average pooling gives shape torch.Size([2, 3, 35, 60])
Global average pooling gives tensor([[[[0.6434]],

         [[0.5972]],

         [[0.5825]]],


        [[[0.7631]],

         [[0.2601]],

         [[0.1085]]]]).


### 2. Basic CNN for Fashion MNIST

In [9]:
import torch 
import torchvision
import torchvision.transforms.v2 as T 

from torch.utils.data import DataLoader

# Let us first insert the dataset preparation code from ImageClassifier.ipynb
torch.manual_seed(42)

toTensor = T.Compose([T.ToImage(), T.ToDtype(torch.float32, scale=True)])
train_and_valid_data = torchvision.datasets.FashionMNIST(root="datasets", train=True, download=True, transform=toTensor)
train_data, val_data = torch.utils.data.random_split(train_and_valid_data, [55_000, 5_000])

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=32)

test_data = torchvision.datasets.FashionMNIST(root="datasets", train=False, download=True, transform=toTensor)
test_loader = DataLoader(test_data, batch_size=32)

100%|██████████| 26.4M/26.4M [00:02<00:00, 11.1MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 2.40MB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 10.7MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 10.3MB/s]


In [11]:
from functools import partial
import torchmetrics
from HelperMethods import HelperML


# Provide default values for convolutional layer
DefaultConv2d = partial(nn.Conv2d, kernel_size=3, padding="same", bias=False)

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

model = nn.Sequential(
    DefaultConv2d(in_channels=1, out_channels=64, kernel_size=7), 
    nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(kernel_size=2), 
    DefaultConv2d(in_channels=64, out_channels=128),
    nn.BatchNorm2d(128), nn.ReLU(),
    DefaultConv2d(in_channels=128, out_channels=128),
    nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(kernel_size=2),
    DefaultConv2d(in_channels=128, out_channels=256),
    nn.BatchNorm2d(256), nn.ReLU(),
    DefaultConv2d(in_channels=256, out_channels=256),
    nn.BatchNorm2d(256), nn.ReLU(), nn.MaxPool2d(kernel_size=2),
    nn.Flatten(),
    nn.Linear(in_features=2304, out_features=128, bias=False), 
    nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.5),
    nn.Linear(in_features=128, out_features=64, bias=False),
    nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.5),
    nn.Linear(in_features=64, out_features=10)
).to(device)

n_epochs=20
bn_momentum=0.01 
lr = 0.05

optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, nesterov=True)
cosine_repeat_scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=2, T_mult=2, eta_min=0.001)

criterion = nn.CrossEntropyLoss()
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)

helperML = HelperML(device)
helperML.train(model, optimizer, criterion, accuracy, train_loader, val_loader, n_epochs, cosine_repeat_scheduler)


Epoch 1/20,
train_loss: 0.5771,
train_metric: 0.8099,
valid_metric: 0.8726.

Epoch 2/20,
train_loss: 0.3650,
train_metric: 0.8826,
valid_metric: 0.8982.

Epoch 3/20,
train_loss: 0.3121,
train_metric: 0.8983,
valid_metric: 0.9094.

Epoch 4/20,
train_loss: 0.2834,
train_metric: 0.9084,
valid_metric: 0.9150.

Epoch 5/20,
train_loss: 0.2583,
train_metric: 0.9151,
valid_metric: 0.9124.

Epoch 6/20,
train_loss: 0.2393,
train_metric: 0.9221,
valid_metric: 0.9238.

Epoch 7/20,
train_loss: 0.2183,
train_metric: 0.9296,
valid_metric: 0.9202.

Epoch 8/20,
train_loss: 0.2012,
train_metric: 0.9346,
valid_metric: 0.9238.

Epoch 9/20,
train_loss: 0.1859,
train_metric: 0.9394,
valid_metric: 0.9252.

Epoch 10/20,
train_loss: 0.1677,
train_metric: 0.9450,
valid_metric: 0.9236.

Epoch 11/20,
train_loss: 0.1620,
train_metric: 0.9486,
valid_metric: 0.9328.

Epoch 12/20,
train_loss: 0.1456,
train_metric: 0.9533,
valid_metric: 0.9260.

Epoch 13/20,
train_loss: 0.1312,
train_metric: 0.9583,
valid_metric: 0.93

{'train_losses': [0.5771217254868699,
  0.3649642229981725,
  0.3121004077606445,
  0.28344054254657175,
  0.2582501972819082,
  0.23927181774946477,
  0.21826773316400594,
  0.2011746467086267,
  0.18591121416360165,
  0.16767409706890254,
  0.1620460830928477,
  0.14560611857184755,
  0.13124533816531395,
  0.12088478630930923,
  0.10860219672311437,
  0.10277829821321219,
  0.0925484895712949,
  0.08284733230032905,
  0.07753876002310786,
  0.068552792218865],
 'train_metrics': [0.8099272847175598,
  0.8825636506080627,
  0.8982545733451843,
  0.9083999991416931,
  0.9151272773742676,
  0.922145426273346,
  0.9296363592147827,
  0.9345636367797852,
  0.9394181966781616,
  0.9449636340141296,
  0.9486363530158997,
  0.9533272981643677,
  0.9582909345626831,
  0.9615636467933655,
  0.9657272696495056,
  0.9671090841293335,
  0.9718363881111145,
  0.9746181964874268,
  0.9762908816337585,
  0.9794363379478455],
 'valid_metrics': [0.8726000189781189,
  0.8981999754905701,
  0.9093999862